In [1]:
from extract import extract_graph , extract_die_area , load_file_content

# Define your file paths
file_name = "aes_run_20260305_181833"
def_path = f"./CTS-Bench/runs/{file_name}/11-openroad-detailedplacement/aes.def"
saif_path = f"./CTS-Bench/runs/{file_name}/aes.saif"
timing_path = f"./CTS-Bench/runs/{file_name}/timing_paths.csv"

# One function call to get everything!
graph = extract_graph(def_path, saif_path, timing_path, clock_port="clk")
def_text = load_file_content(def_path)
die_x_min, die_y_min, die_x_max, die_y_max = extract_die_area(def_text)



print("\nExtraction Complete!")
print(f"Nodes: {len(graph['nodes'])}")
print(f"Skip edges: {graph['skip_edges'].shape}")
print(f"Directed edges: {graph['directed_edges'].shape}")
print(f"Undirected edges: {graph['undirected_edges'].shape}")
print(f"Die area: x[{die_x_min}, {die_x_max}], y[{die_y_min}, {die_y_max}]")    

for a, b in graph['skip_edges'][:5]:  # Print first 5 skip edges
    print(f"Skip edge: {a, b}")

Building cell vocabulary from sky130_fd_sc_hd_tt_025C_1v80.lib...
Vocabulary: 428 unique cells
Flip-flops: 2994
Unique pairs: 5596, dropped: 0
Skip edges shape: (5596, 2)
Nodes: 24364
Directed edges: 77018
Undirected edges: 154036
Flip-flops: 2994
Fan-in  — max: 6, avg: 3.2
Fan-out — max: 153, avg: 3.2

Extraction Complete!
Nodes: 24364
Skip edges: (5596, 2)
Directed edges: (77018, 2)
Undirected edges: (154036, 2)
Die area: x[0, 540775], y[0, 1081235]
Skip edge: (np.int64(20751), np.int64(23414))
Skip edge: (np.int64(20751), np.int64(23415))
Skip edge: (np.int64(20751), np.int64(23412))
Skip edge: (np.int64(20751), np.int64(23452))
Skip edge: (np.int64(20751), np.int64(23605))


In [2]:
import numpy as np 
import torch

def normalize_features(nodes, die_x_min, die_y_min, die_x_max, die_y_max):
    die_w = die_x_max - die_x_min
    die_h = die_y_max - die_y_min
    
    cell_ids = []
    numeric = []
    
    for n in nodes:
        row = [
            (n['x'] - die_x_min) / die_w,
            (n['y'] - die_y_min) / die_h,
            n['dist_to_boundaries'][0] / die_w,
            n['dist_to_boundaries'][1] / die_w,
            n['dist_to_boundaries'][2] / die_h,
            n['dist_to_boundaries'][3] / die_h,
            np.log1p(n['cell_area']),
            n['avg_pin_cap'] * 1000,
            n['total_pin_cap'] * 1000,
            np.log2(max(n['drive_strength'], 1)),
            float(n['is_sequential']),
            float(n['is_buffer']),
            n['toggle_count'],
            n['sum_toggle_count'],
            n['signal_prob'],
            float(n['non_zero_count']),
            np.log1p(n['fan_in']),
            np.log1p(n['fan_out']),
        ]
        numeric.append(row)
        cell_ids.append(n['cell_type_id'])
    
    features = torch.tensor(numeric, dtype=torch.float32)
    cell_type_ids = torch.tensor(cell_ids, dtype=torch.long)
    
    # Standardize all columns except binary flags (indices 10, 11)
    binary_cols = {10, 11}
    norm_stats = {}
    
    for col in range(features.shape[1]):
        if col in binary_cols:
            continue
        mean = features[:, col].mean()
        std = features[:, col].std()
        if std > 1e-8:
            features[:, col] = (features[:, col] - mean) / std
            norm_stats[col] = (mean.item(), std.item())
    
    return features, cell_type_ids, norm_stats

# Assuming 'graph' and 'die_x_min' etc. are already defined from previous cells
X_float, X_cell_ids, norm_stats = normalize_features(
    graph['nodes'], 
    die_x_min, die_y_min, die_x_max, die_y_max
)

print("-" * 30)
print("NORMALIZATION RESULTS")
print("-" * 30)
print(f"X_float shape:    {X_float.shape} (Standardized continuous features)")
print(f"X_cell_ids shape: {X_cell_ids.shape} (Categorical IDs for embeddings)")
print(f"Number of normalized columns: {len(norm_stats)}")

# Sample check on first node
print("\nFirst Node Normalized Features:")
print(X_float[17])
print(X_cell_ids[17])


# Verify the categorical ID
print(f"\nFirst Node Cell Type ID: {X_cell_ids[0].item()}")
print(f"\nSecond Node Cell Type ID: {X_cell_ids[1045].item()}")

------------------------------
NORMALIZATION RESULTS
------------------------------
X_float shape:    torch.Size([24364, 18]) (Standardized continuous features)
X_cell_ids shape: torch.Size([24364]) (Categorical IDs for embeddings)
Number of normalized columns: 16

First Node Normalized Features:
tensor([-0.5958,  1.0806, -0.5958,  0.5958, -1.0806,  1.0806,  0.7999,  8.2356,
         1.6574,  3.3326,  0.0000,  1.0000,  0.9242,  0.8937,  0.4033, -1.5944,
        -2.3439,  3.2434])
tensor(199)

First Node Cell Type ID: 201

Second Node Cell Type ID: 20


In [3]:
import scipy.sparse as sp
import numpy as np

def build_X_hop_mask(n_nodes, undirected_edges, hop_mask_len=3):
    """
    Creates the Omega mask. 
    If Omega[i, j] = 1, Node i is allowed to 'use' Node j for reconstruction.
    """
    # 1. Slice out the Source (rows) and Destination (cols) nodes
    u_rows = undirected_edges[:, 0]
    u_cols = undirected_edges[:, 1]
    
    # 2. Build the initial 1-hop Sparse Adjacency Matrix (A)
    # Using bool to save memory (we only care IF a wire exists)
    A = sp.csr_matrix((np.ones(len(u_rows), dtype=bool), (u_rows, u_cols)), 
                      shape=(n_nodes, n_nodes))
    
    # 3. Matrix Power Expansion (Find 2-hop and 3-hop neighbors)
    omega = A.copy()
    temp = A.copy()
    
    for i in range(2, hop_mask_len + 1):
        temp = temp.dot(A)
        omega = omega + temp
        print(f"  -> Hop {i} expansion complete.")

    # 4) diag(P) = 0
    # A node cannot reconstruct itself
    omega.setdiag(0)
    omega.eliminate_zeros()
    
    # 5. Convert back to coordinates for PyTorch
    omega_coo = omega.tocoo()
    
    print(f"Mask created! {omega_coo.nnz:,} total connections allowed.")
    return omega_coo.row, omega_coo.col


# Pass the length and the edges explicitly 
p_rows, p_cols = build_X_hop_mask(
    n_nodes=len(graph['nodes']), 
    undirected_edges=graph['undirected_edges'], 
    hop_mask_len=3
)


# # 1. Create a reverse dictionary to turn indices back into DEF instance names
# idx_to_node = {idx: name for name, idx in graph['node_to_idx'].items()}

# # 2. Check the overall density (How heavy is this for the GPU?)
# total_edges = len(p_rows)
# total_nodes = len(graph['nodes'])
# avg_neighbors = total_edges / total_nodes

# print("--- MASK STATISTICS ---")
# print(f"Total Nodes (Gates): {total_nodes:,}")
# print(f"Total Allowed Connections in Mask: {total_edges:,}")
# print(f"Average 3-Hop Neighbors per Gate: {avg_neighbors:.1f}")

# # 3. Peek at the actual relationships (The first 10 connections)
# print("\n--- SAMPLE CONNECTIONS (Who can see who?) ---")
# for i in range(10):
#     target_idx = p_rows[i]
#     neighbor_idx = p_cols[i]
    
#     target_name = idx_to_node[target_idx]
#     neighbor_name = idx_to_node[neighbor_idx]
    
#     print(f"Target Gate [{target_idx:5}]: {target_name:<20} <-- looks at --> Neighbor [{neighbor_idx:5}]: {neighbor_name}")

# # 4. Look at a specific gate's neighborhood
# sample_gate_idx = p_rows[0] # Let's pick whatever gate happens to be first
# sample_gate_name = idx_to_node[sample_gate_idx]

# # Find all neighbors for this specific gate
# neighbors_of_sample = p_cols[p_rows == sample_gate_idx]

# print(f"\n--- SPOTLIGHT: Gate '{sample_gate_name}' ---")
# print(f"This gate has {len(neighbors_of_sample)} neighbors within 3 hops.")
# print(f"First 5 neighbors it will use for reconstruction:")
# for n_idx in neighbors_of_sample[:5]:
#     print(f"  - {idx_to_node[n_idx]}")



  -> Hop 2 expansion complete.
  -> Hop 3 expansion complete.
Mask created! 10,922,930 total connections allowed.


In [4]:
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F

class FirstTerm(nn.Module):
    def __init__(self, indices, num_nodes , num_cell_types, num_of_clusters, embedding_dim=8):
        super().__init__()
        self.register_buffer('indices', indices)
        self.cell_embedding = nn.Embedding(num_cell_types, embedding_dim)
        # Initialize 10.9M weights (The Dials)
        self.p_weights = nn.Parameter(torch.randn(indices.shape[1]) * 0.01)
        self.num_nodes = num_nodes

        # 3. THE CLUSTER HEAD (The C-Matrix Generator)
        # This replaces the N x K parameter bottleneck.
        self.cluster_head = nn.Sequential(
            nn.Linear(18 + embedding_dim, 64), 
            nn.ReLU(),
            nn.Linear(64, num_of_clusters)      # Output is K clusters
        )

    def forward(self, X, X_cell_ids):
        cell_features = self.cell_embedding(X_cell_ids)  # Shape: [num_nodes, embedding_dim]
        X_combined = torch.cat([X, cell_features], dim=1)  # Shape: [num_nodes, 18 + embedding_dim]


        # Enforce P >= 0 and build sparse matrix
        P = torch.sparse_coo_tensor(self.indices, torch.relu(self.p_weights)  + 1e-6, 
                                    (self.num_nodes, self.num_nodes)).coalesce()
        
        # Reconstruction: XP
        X_hat = torch.sparse.mm(P, X)
        
        # Loss: ||X - XP||
        error = X - X_hat
        loss1 = torch.sum(error ** 2)

        # Pass all node features through the head
        logits = self.cluster_head(X_combined) # Shape: [n, k]
        
        #TERM2
        #C matrix with probability distribution across clusters for each node
        temperature = 0.2
        C = F.softmax(logits/temperature, dim=-1) # Shape: [n, k]

        c_vals = P.values()
        c_vals_norm = c_vals / (c_vals.max().detach() + 1e-8)
        
        # Rebuild using the exact same sorted indices
        P_norm = torch.sparse_coo_tensor(P.indices(), c_vals_norm, 
                                         (self.num_nodes, self.num_nodes)).coalesce()
        
        # 2. Convert to CSR format (Required for the CUDA SDDMM engine)
        P_csr = P_norm.to_sparse_csr()
        
        # 3. SDDMM Magic! 
        # beta=1.0, alpha=-1.0 calculates exactly: (1.0 * P_csr) - (1.0 * C @ C^T)
        # It ONLY calculates this at the 10.9M non-zero locations!
        diff_csr = torch.sparse.sampled_addmm(P_csr, C, C.t(), beta=1.0, alpha=-1.0)
        
        # 4. Square the differences and sum them
        loss2 = torch.sum(diff_csr.values() ** 2)


        # Term 2 is naturally smaller than Term 1. 
        # We multiply it by 50 to make the clustering "matter" to the optimizer.
        alpha = 1
        loss = loss1 + (alpha * loss2)
        



        return loss , loss1 , loss2,  C

# --- 3. THE INTEGRATED EXECUTION (The Run) ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_nodes = len(graph['nodes'])

# Step A: Build the mask
# 1. Get the two lists (rows and columns) from your function
rows, cols = build_X_hop_mask(num_nodes, graph['undirected_edges'], hop_mask_len=3)

# 2. Stack them into one Tensor and then move to device
p_indices = torch.tensor(np.array([rows, cols]), dtype=torch.long).to(device)

num_cell_types = int(X_cell_ids.max().item() + 1)

n_ff = (X_float[:, 10] == 1.0).sum().item()
num_of_clusters = int(n_ff // 2)
num_of_clusters = max(1, num_of_clusters)

# Step B: Initialize Model & Data
model = FirstTerm(p_indices, num_nodes, num_cell_types , num_of_clusters).to(device)
X_tensor = X_float.to(device) # Your normalized features
cell_ids_tensor = X_cell_ids.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.01)



# Step C: The Optimization Loop
print(f"\nTraining on {device}...")

for epoch in range(1001):
    optimizer.zero_grad()

    # Unpack the loss and the clustering matrix C
    loss, loss1, loss2, C = model(X_tensor, cell_ids_tensor)

    loss.backward()
    optimizer.step()
    
    if epoch % 10 == 0:
        # Calculate the "peakiness" of your clusters for monitoring
        avg_max_prob = C.max(dim=1)[0].mean().item()
        print(
            f"Epoch {epoch:2} | "
            f"Total Loss: {loss.item():.4f} | "
            f" Loss1: {loss1.item():.4f} | "
            f"Loss2: {loss2.item():.4f} | "
            f"Avg Confidence: {avg_max_prob:.2%}"
        )

print("\nSuccess! Your manifold is trained. Your chip data is now organized.")

# --- Printing C results ---
print("\nSample Clustering Matrix (C) - First 5 Nodes:")
# Each row represents a node's probability distribution across k clusters [cite: 109, 110]
print(C[:5].detach().cpu().numpy()) 

print(f"\nMatrix C Shape: {C.shape} (Nodes x Clusters)")
print(f"Verification - Sum of Row 0: {C[0].sum().item():.2f}")

  -> Hop 2 expansion complete.
  -> Hop 3 expansion complete.
Mask created! 10,922,930 total connections allowed.

Training on cuda...


/tmp/ipykernel_1186657/3833783362.py:54: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  P_csr = P_norm.to_sparse_csr()


Epoch  0 | Total Loss: 626399.7500 |  Loss1: 435159.8125 | Loss2: 191239.9375 | Avg Confidence: 2.27%
Epoch 10 | Total Loss: 82613.8281 |  Loss1: 45581.0898 | Loss2: 37032.7422 | Avg Confidence: 17.35%
Epoch 20 | Total Loss: 44654.9297 |  Loss1: 26107.8711 | Loss2: 18547.0586 | Avg Confidence: 17.40%
Epoch 30 | Total Loss: 29907.6699 |  Loss1: 17948.3594 | Loss2: 11959.3105 | Avg Confidence: 15.27%
Epoch 40 | Total Loss: 22678.4785 |  Loss1: 14328.1982 | Loss2: 8350.2803 | Avg Confidence: 14.46%
Epoch 50 | Total Loss: 18574.1875 |  Loss1: 12194.2480 | Loss2: 6379.9385 | Avg Confidence: 13.61%
Epoch 60 | Total Loss: 16122.1348 |  Loss1: 10846.7930 | Loss2: 5275.3418 | Avg Confidence: 13.18%
Epoch 70 | Total Loss: 14340.2480 |  Loss1: 9935.0000 | Loss2: 4405.2480 | Avg Confidence: 12.82%
Epoch 80 | Total Loss: 13112.7832 |  Loss1: 9267.5898 | Loss2: 3845.1938 | Avg Confidence: 12.55%
Epoch 90 | Total Loss: 12237.3301 |  Loss1: 8763.8477 | Loss2: 3473.4829 | Avg Confidence: 12.44%
Epoch 1